In [1]:
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [12]:
from peft import LoftQConfig, LoraConfig, CordaConfig

ImportError: cannot import name 'CordaConfig' from 'peft' (/home/omar-elmasaoudi/miniconda3/envs/ml_dev_env/lib/python3.10/site-packages/peft/__init__.py)


### CorDA Module

Given our task of enabling Qwen models to perform better on specific downstream tasks, a common issue with QLoRA/LoRA and other PEFT methods is that these modules often forget the context of the downstream task and may also overwrite pre-trained world knowledge (stored in the parameters of the base model).
To address this, we add an additional module to our fine-tuning pipeline called CorDA.



### What is CorDA

CorDA, short for Context-Oriented Decomposition Adaptation, decomposes each linear layer’s weight matrix into subspaces that reflect the contextual importance of features for a given task.
This allows us to separate task-specific components (directions strongly activated for the target task) from generalizable components (directions associated with pre-trained knowledge).

We obtain these subspaces through the following process:

1. **Collect activations:**
   For a specified task (e.g., QA or another evaluation dataset), pass representative samples $(x, \hat{y})$ through the model and record the pre-activation hidden outputs from each linear layer:
   $$
   X \in \mathbb{R}^{N \times D}
   $$
   where $N$ is the number of tokens in the batch and $D$ is the hidden dimension.

2. **Compute the feature covariance matrix:**
   $$
   \Sigma_X = X^T X
   $$
   This covariance matrix captures how different neurons (features) co-activate across samples, revealing which parts of the representation space are most relevant to the task context.

3. **Compute the context-oriented weight matrix:**
   $$
   A = W \Sigma_X
   $$
   where $W \in \mathbb{R}^{D_{\text{out}} \times D_{\text{in}}}$.
   This step reorients the layer’s weights toward the input directions that are most active in the dataset’s context.

4. **Perform Singular Value Decomposition (SVD):**
   Decompose $A$ as:
   $$
   A = U S V^T
   $$
   where the diagonal matrix $S$ contains singular values $\sigma_1, \sigma_2, \dots$ that rank the importance of each component.

   * Large singular values capture dominant, task-relevant directions.
   * Smaller singular values capture more general, residual directions.

   The top-$r$ components (largest singular values) represent highly task-specific subspaces, while the smallest-$r$ components correspond to more generalizable pathways.

5. **Selective adaptation:**
   CorDA allows us to choose which subspace to make trainable:

   * *Task-specialized mode:* Freeze the small-$\sigma$ components and fine-tune the large-$\sigma$ ones.
     → Maximizes performance on the downstream task but may cause catastrophic forgetting.
   * *Knowledge-preserved mode:* Freeze the large-$\sigma$ components and fine-tune the small-$\sigma$ ones.
     → Retains general world knowledge while improving task performance with minimal forgetting.




In [ ]:
@torch.no_grad()
def run_model():
    model.eval()
    for batch in dataset:
        model(**batch)
        
        
corda_config = CordaConfig(
    
)

In [3]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    dtype="auto",
    device_map="auto",
    use_safetensors=True   
)

In [5]:
from peft import get_peft_model

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()



trainable params: 1,572,864 || all params: 332,769,280 || trainable%: 0.4727


/home/omar-elmasaoudi/miniconda3/envs/ml_dev_env/lib/python3.10/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/omar-elmasaoudi/miniconda3/envs/ml_dev_env/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


### Save the model locally.

In [6]:
lora_model.save_pretrained("tuned_model/opt-350m-lora")

### Load the model for inference

In [7]:
from peft import PeftModel, PeftConfig

config = PeftConfig.from_pretrained("tuned_model/opt-350m-lora")
model = AutoModelForCausalLM.from_pretrained(config.adapter_config)
lora_model = PeftModel.from_pretrained(model, "tuned_model/opt-350m-lora")

AttributeError: 'LoraConfig' object has no attribute 'adapter_config'